# Description

This code generate a co-citation graph of the corpus. The output are in PDF and PGF formats.
Each reference appearing in the corpus are processed: the Levenstein distance is computed to compare reference to one another.

# Prerequisites

- matplotlib
- pandas
- openpyxl
- Levenstein
- [grobid](https://grobid.readthedocs.io/en/latest/Grobid-docker/) to generate the tei.xml files of each reference.


Use the following to install the packages:

```pip install -e ".[citation_network]"```


## A note on 'grobid'

Use the lightweight version to generate the tei.xml file, using the following command:

```
docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.2
```

The server is available on your machine at localhost, port 8070: http://localhost:8070/

In the tab 'TEI', select the 'Process all References' in 'Process to call' and check the option 'Consolidate citation'; select the PDF file of the paper you want to process and then clic on 'Submit'. The resulting file is then saved.

# Input

You need a `filename` in excel format in the `data` folder.
The `RAW` sheet must contain at least the following columns:
- 'Title'
- 'BiblCitation' containing a string with \cite{xxx} in LaTeX format
- 'TeiFile'

# Output 

- Two files named `output_file`.ext, with the following extensions:
    - PGF
    - PDF


In [ ]:
import pandas as pd
from grobid_client import GrobidClient
import reference_man
import random
import numpy as np
import os


In [ ]:
workspace = '../data'
sheet_name = 'RAW'
output_file = "cocitation_graph"
folder_name = "refs"
folder = os.path.join(workspace, folder_name)
file_format = 'pdf' # or 'pdf'

seed = 42
random.seed(seed)
np.random.seed(seed)
levenstein_threshold = 0.75
num_leafs = 50

In [ ]:
# If not xml => pdf
from reference_man import Publication
if file_format == 'pdf':
    # create a folder of xml if not exist
    files_pdf = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
    folder_xml = f"{workspace}/refs_table_xml"
    
    os.makedirs(folder_xml, exist_ok=True)
    client = GrobidClient()
    for filepdf in files_pdf:
        output_path = os.path.join(folder_xml, f"{filepdf}.tei.xml")
        posixpath = client.process_and_save(os.path.join(folder, filepdf), output_path)
        print(posixpath)
        xmlfile = os.path.join(folder_xml, posixpath.name)
        pub = Publication.from_xml(xmlfile)
        
    files = [f for f in os.listdir(folder_xml) if os.path.isfile(os.path.join(folder_xml, f))]

else:
    files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
    folder_xml = folder


In [ ]:
refs = {}
corpus_files = list(files)
print(len(corpus_files))
refs_dict = {}
corpus_titles = []

In [ ]:
for item in corpus_files:
    pub = reference_man.Publication.from_xml(os.path.join(folder_xml, item))
    pub_title = pub.get_title()
    print(item)
    print("\t",pub_title)
    refs[pub_title] = set()
    corpus_titles.append(pub_title)
    for reference in pub.get_references():
        ref_title = reference.get_title()
        if ref_title is not None:
            if ref_title not in refs:
                refs[ref_title] = set()

In [ ]:
import Levenshtein

In [ ]:
ref_keys = list(refs.keys())
ref_keys_lower = [k.lower() for k in ref_keys]

for item in corpus_files:
    pub = reference_man.Publication.from_xml(os.path.join(folder_xml, item))
    pub_title = pub.get_title()
    pub_refs = pub.get_references()

    print("-----", pub_title)
    print("nb references=", len(pub_refs))
    refs_dict[pub_title] = pub_refs

    for reference in pub_refs:
        ref_title = reference.get_title()
        if ref_title is None or ref_title == pub_title:
            continue

        ref_title_lower = ref_title.lower()
        best_ratio = levenstein_threshold
        best_match = None
        for i, t_lower in enumerate(ref_keys_lower):
            ratio = Levenshtein.ratio(t_lower, ref_title_lower)
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = ref_keys[i]
        if best_match:
            refs[pub_title].add(best_match)
            

In [ ]:
use_pfg = True

if use_pfg:
    import matplotlib as mpl

    mpl.use("pgf")
    mpl.rcParams.update({
        "pgf.texsystem": "pdflatex",   
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,    
    })


In [ ]:
corpus = pd.read_excel(os.path.join(workspace, "refs.xlsx"), sheet_name=sheet_name)

In [ ]:
from matplotlib import pyplot as plt
import networkx as nx

G = nx.Graph()

corpus_files = set(refs.keys())

for id,item in corpus.iterrows():
    doc = item['Title']
    tei_name = item['TeiFile']
    references = refs.get(tei_name)
    if references is None:
        continue
    label_name = item["BiblCitation"]
    print(label_name)
    G.add_node(doc, type="corpus",label=label_name)

    for ref in references:
        if ref != doc:
            G.add_edge(doc, ref, weight=1)

labels = nx.get_node_attributes(G, "label")


print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())



In [ ]:
for corpus_node in [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]:
    neighbors = list(G.neighbors(corpus_node))

    leaf_neighbors = [n for n in neighbors if G.degree[n] == 1 and G.nodes[n].get("type") != "corpus"]

    keep = set(leaf_neighbors[:num_leafs])

    remove = [n for n in leaf_neighbors if n not in keep]
    G.remove_nodes_from(remove)

print("Graph :", G.number_of_nodes(), "nodes and ", G.number_of_edges(), " edges")

In [ ]:
low_degree_nodes = [n for n, d in G.degree() if d <= 1]
G.remove_nodes_from(low_degree_nodes)
print("Final graph :", G.number_of_nodes(), "nodes and ", G.number_of_edges(), " edges")

In [ ]:
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, seed=seed, k=0.55, iterations=150)

red_edges = [
    (u, v) for u, v in G.edges()
    if G.nodes[u].get("type") == "corpus" and G.nodes[v].get("type") == "corpus"
]
gray_edges = [
    (u, v) for u, v in G.edges()
    if (u, v) not in red_edges
]

nx.draw_networkx_edges(G, pos, edgelist=gray_edges, edge_color="gray",
                       alpha=0.4, width=0.6, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

nx.draw_networkx_edges(G, pos, edgelist=red_edges, edge_color="red",
                       width=2, alpha=0.8, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

corpus_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]
ref_nodes    = [n for n, d in G.nodes(data=True) if d.get("type") != "corpus"]

nx.draw_networkx_nodes(G, pos, nodelist=ref_nodes, node_size=1, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=corpus_nodes, node_color="red", node_size=10, ax=ax)

texts = []
for node, label in labels.items():
    x, y = pos[node]
    t = ax.text(x, y, label, fontsize=12, va="bottom", ha="left")
    texts.append(t)

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.4)
)

In [ ]:

plt.savefig(output_file+".pgf")  
plt.savefig(output_file+".pdf")  
